# 10 · Memory & Context — with a Gradio UI

**Where we are in the stack:** the **session layer**. Notebook 01 proved the model is
stateless - "memory" is just re-feeding the transcript. Notebook 02's agent held state only
*within* one run. Today: a session that spans **many turns**, and what to do when the
transcript outgrows the context window.

Networking has this exact split: **UDP vs TCP**. Each API call is a datagram - the model
remembers nothing. A *session* is TCP: **your** code keeps the connection state (the
transcript), retransmits it with every message, and tears it down or checkpoints it.

Three problems, three mechanisms:

| Problem | Mechanism | Network analogue |
|---|---|---|
| The model forgets between turns | keep + re-send the transcript | connection state |
| The transcript outgrows the window | **summarize** old turns into a note | route aggregation |
| The kernel dies / session moves | **checkpoint** to disk, resume | graceful restart / state sync |

Tools and loop are the ones from notebook 02, unchanged.

> Needs a tool-capable model (see notebook 02).

In [ ]:
# --- Provider config: works with OpenAI, OpenRouter, or a local OpenAI-compatible server ---
import os
from openai import OpenAI

# Load settings from a .env file if present (falls back to existing env vars).
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    import os
    if os.path.exists(".env"):
        for _line in open(".env"):
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())


# Pick ONE setup by exporting these env vars before launching Jupyter.
#
#   OpenAI:     OPENAI_BASE_URL=https://api.openai.com/v1   MODEL=gpt-4o-mini
#   OpenRouter: OPENAI_BASE_URL=https://openrouter.ai/api/v1 MODEL=openai/gpt-4o-mini
#   Local:      OPENAI_BASE_URL=http://localhost:11434/v1    MODEL=qwen2.5:7b   (Ollama)
#               (use 'qwen2.5' / 'llama3.1' etc. - a 1B model is great for chat but
#                usually too weak to drive tool-calling reliably.)

BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
API_KEY  = os.environ.get("OPENAI_API_KEY", "set-me")   # any non-empty string for local servers
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

# Behind a TLS-intercepting firewall/proxy, HTTPS cert verification can fail.
# Set VERIFY_SSL=false in .env to skip it: we hand the OpenAI SDK a custom
# httpx client with verification turned off. Leave it true everywhere else.
import httpx
VERIFY_SSL = os.environ.get("VERIFY_SSL", "true").strip().lower() not in ("false", "0", "no")
http_client = httpx.Client(verify=VERIFY_SSL)
if not VERIFY_SSL:
    import warnings
    warnings.filterwarnings("ignore")
    print("\u26a0\ufe0f  SSL verification DISABLED (VERIFY_SSL=false) \u2014 use only on a trusted network")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY, http_client=http_client)
print("endpoint:", BASE_URL, "| model:", MODEL)

## 1. The tools (verbatim from notebook 02)

Nothing new here - subnet math and (mocked) interface telemetry. The lesson today is in the
*session* wrapped around them, not the tools.

In [ ]:
import ipaddress, hashlib, json

def calculate_subnet(cidr):
    '''Compute network, broadcast, netmask and usable host count for a CIDR.'''
    net = ipaddress.ip_network(cidr, strict=False)
    if net.version == 4:
        usable = net.num_addresses - 2 if net.prefixlen <= 30 else net.num_addresses
    else:
        usable = net.num_addresses
    return {
        "network": str(net.network_address),
        "broadcast": str(net.broadcast_address) if net.version == 4 else "n/a",
        "netmask": str(net.netmask),
        "prefix_length": net.prefixlen,
        "total_addresses": net.num_addresses,
        "usable_hosts": usable,
    }

def get_interface_status(device, interface):
    '''MOCK telemetry. In production wire this to netmiko / SNMP / gNMI / your MCP server.'''
    h = int(hashlib.md5(f"{device}{interface}".encode()).hexdigest(), 16)
    up = (h % 5 != 0)  # ~80 percent up, deterministic so demos are repeatable
    return {
        "device": device, "interface": interface,
        "admin_status": "up",
        "oper_status": "up" if up else "down",
        "speed": "10Gbps", "mtu": 1500,
        "input_errors": h % 7, "output_errors": h % 3, "crc_errors": h % 4,
    }

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "calculate_subnet",
        "description": "Compute network, broadcast, netmask and usable host count for an IPv4/IPv6 CIDR.",
        "parameters": {"type": "object",
            "properties": {"cidr": {"type": "string", "description": "CIDR, e.g. 10.20.0.0/22"}},
            "required": ["cidr"]}}},
    {"type": "function", "function": {
        "name": "get_interface_status",
        "description": "Operational status and error counters for an interface on a device.",
        "parameters": {"type": "object",
            "properties": {
                "device":    {"type": "string", "description": "hostname, e.g. leaf-01"},
                "interface": {"type": "string", "description": "interface, e.g. ethernet1/0/1"}},
            "required": ["device", "interface"]}}},
]

# name -> callable. This is your capability table.
TOOL_REGISTRY = {
    "calculate_subnet": calculate_subnet,
    "get_interface_status": get_interface_status,
}

## 2. A session: the transcript IS the memory

`Session.ask()` is notebook 02's FSM with one change: `self.history` **persists between
calls** instead of being rebuilt from scratch. Every turn - user text, tool calls, tool
results, answers - is appended and re-sent. That is the entire trick behind every "chat
with memory" product you have used.

In [ ]:
import json

SYSTEM = ("You are a network operations assistant working a ticket with the user. "
          "Use tools when they help. Remember facts the user tells you. Be concise.")

class Session:
    """A multi-turn agent session. The transcript is the memory."""

    def __init__(self):
        self.history = [{"role": "system", "content": SYSTEM}]

    def ask(self, user_text, max_iterations=5, temperature=0):
        self.history.append({"role": "user", "content": user_text})
        print("USER:", user_text); print("=" * 64)

        for step in range(1, max_iterations + 1):
            resp = client.chat.completions.create(
                model=MODEL, messages=self.history, tools=TOOLS, temperature=temperature,
            )
            msg = resp.choices[0].message

            # TERMINATION: no tool requested -> final answer (stays in history).
            if not msg.tool_calls:
                self.history.append({"role": "assistant", "content": msg.content})
                print(f"[step {step}] ANSWER: {msg.content}\n")
                return msg.content

            self.history.append({
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [
                    {"id": tc.id, "type": "function",
                     "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                    for tc in msg.tool_calls
                ],
            })
            for tc in msg.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments)
                print(f"[step {step}] TOOL  -> {name}({args})")
                try:
                    result = TOOL_REGISTRY[name](**args)     # <-- WE run it
                except Exception as e:
                    result = {"error": str(e)}
                print(f"[step {step}] RESULT <- {str(result)[:160]}")
                self.history.append({
                    "role": "tool", "tool_call_id": tc.id,
                    "content": json.dumps(result),
                })

        return "Stopped: hit max_iterations (TTL expired)."

## 3. Run a multi-turn troubleshooting session

Turn 2 only works because turn 1's facts (the ticket number, the tool results) are still in
`history`. Re-run turn 2 in a *fresh* `Session()` to watch it fail, exactly like notebook
01's forgotten name.

In [ ]:
s = Session()
_ = s.ask("We're working ticket NOC-1042. Check whether ethernet1/0/1 on leaf-01 is up.")

In [ ]:
_ = s.ask("Which ticket is this again - and did that interface show any CRC errors?")
print("messages in history:", len(s.history))

## 4. The context budget: summarize old turns (route aggregation)

The window is finite and priced per token; a long session eventually will not fit. Naive fix:
drop the oldest turns (a FIFO queue) - cheap, but the facts in them vanish. Better:
**aggregate the routes** - collapse many old messages into one short system note, keeping
the *facts* while discarding the *transcript*.

Trade-off to be honest about: summarization is lossy, and it costs one extra model call.
What survives is whatever the summarizer thought mattered - pin anything critical (ticket
IDs, decisions) explicitly in the prompt if you cannot afford to lose it.

In [ ]:
def compact(session, keep_last=2):
    """Collapse everything but the last `keep_last` exchanges into one memory note."""
    head, tail = session.history[1:-keep_last], session.history[-keep_last:]
    if not head:
        return
    resp = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[
            {"role": "system", "content":
                "Summarize this network-ops conversation into a short factual memory note. "
                "Keep: ticket IDs, devices, interfaces, tool findings, decisions. "
                "Drop: pleasantries and step-by-step narration."},
            {"role": "user", "content": json.dumps(head)},
        ],
    )
    note = resp.choices[0].message.content
    session.history = (
        [session.history[0],                                  # original system prompt
         {"role": "system", "content": "Memory of earlier turns:\n" + note}]
        + tail
    )
    print("MEMORY NOTE:\n" + note)

before = len(s.history)
compact(s)
print(f"\nhistory: {before} messages -> {len(s.history)}")

### Prove the memory survived compaction

In [ ]:
_ = s.ask("Remind me: which device and interface are we looking at for this ticket?")

## 5. Checkpoint & resume (graceful restart)

The session state is just a list of dicts - so persistence is `json.dump`. Save, kill the
kernel, come back tomorrow, `load()` and continue mid-investigation. This is precisely what
LangGraph's *checkpointing* (the feature list from notebook 02's recap) automates - plus
versioning, threads, and stores - and now you have hand-rolled its core.

The checkpoint file is **git-ignored**: transcripts routinely contain things that must not
land in a repo.

In [ ]:
CHECKPOINT = ".agent_checkpoint.json"   # git-ignored

def save(session, path=CHECKPOINT):
    with open(path, "w") as f:
        json.dump(session.history, f, indent=2)
    print(f"saved {len(session.history)} messages -> {path}")

def load(path=CHECKPOINT):
    restored = Session()
    with open(path) as f:
        restored.history = json.load(f)
    print(f"restored {len(restored.history)} messages <- {path}")
    return restored

save(s)
s2 = load()          # pretend this is a new kernel, hours later
_ = s2.ask("Given everything so far, what should the next step on this ticket be?")

## Recap

Memory is not a model feature - it is **session engineering**, and you just built all of it:

- **Transcript persistence**: the model is UDP; your `Session` object is the TCP state.
- **Summarization**: route aggregation for context - keep the facts, drop the packets.
  Lossy by nature; pin what you cannot afford to lose.
- **Checkpointing**: state is data; `json.dump` is a graceful restart.

Production systems layer more on top - per-user long-term memory in a database, RAG over
past conversations (notebook 07's index, pointed at transcripts), framework checkpointers
(notebook 03) - but they are all compositions of these three mechanisms.

**Next:** many sessions, many specialists - agents calling agents. ->
`11_multi_agent_orchestration.ipynb`

## 6. Interactive UI (Gradio)

A chat box wired to one persistent `Session`, with live message-count so you can watch the
transcript grow - plus buttons for the two session operations: **Compact** (summarize old
turns) and **Save / Load** checkpoint.

In [ ]:
# Install Gradio for the interactive UI (run once).
%pip install -q gradio

In [ ]:
import io, contextlib

def _run_capture(fn, *args, **kwargs):
    """Run an agent function, capturing its printed step-by-step trace."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        try:
            result = fn(*args, **kwargs)
        except Exception as e:
            result = f"Error: {e}"
    return buf.getvalue(), (result or "")

In [ ]:
import gradio as gr

ui_session = Session()

def chat_ui(user_text, chat_log):
    trace, answer = _run_capture(ui_session.ask, user_text)
    chat_log = chat_log + [(user_text, answer)]
    return chat_log, "", trace, f"{len(ui_session.history)} messages in history"

def compact_ui():
    trace, _ = _run_capture(compact, ui_session)
    return trace, f"{len(ui_session.history)} messages in history"

def save_ui():
    trace, _ = _run_capture(save, ui_session)
    return trace, f"{len(ui_session.history)} messages in history"

def load_ui():
    global ui_session
    trace, restored = _run_capture(load)
    if not isinstance(restored, str):
        ui_session = restored
    return trace, f"{len(ui_session.history)} messages in history"

with gr.Blocks(title="10 · Memory & Context") as demo:
    gr.Markdown("# 🧠 Multi-turn Network Ops Session\n"
                "One persistent session: chat, watch the transcript grow, compact it, "
                "checkpoint it, restore it.")
    chat_log = gr.Chatbot(label="Session")
    user_text = gr.Textbox(
        label="Your message",
        value="We're working ticket NOC-1042. Check whether ethernet1/0/1 on leaf-01 is up.",
    )
    with gr.Row():
        send = gr.Button("Send", variant="primary")
        compact_btn = gr.Button("Compact history")
        save_btn = gr.Button("Save checkpoint")
        load_btn = gr.Button("Load checkpoint")
    counter = gr.Markdown("1 messages in history")
    trace = gr.Textbox(label="Agent trace / session ops", lines=10)
    send.click(chat_ui, inputs=[user_text, chat_log],
               outputs=[chat_log, user_text, trace, counter])
    compact_btn.click(compact_ui, outputs=[trace, counter])
    save_btn.click(save_ui, outputs=[trace, counter])
    load_btn.click(load_ui, outputs=[trace, counter])

demo.launch()